In [ ]:
# A: setup -- engine at the notebook's pinned commit, the GGUF loader module
import os, sys, time, json, pathlib, subprocess, threading, gc
T0 = time.perf_counter()
WORK = pathlib.Path("/content/gguf-exp"); WORK.mkdir(parents=True, exist_ok=True)
for p in ("safetensors", "tokenizers"):
    __import__("importlib").util.find_spec(p) or subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=True)
ENGINE_DIR = WORK / "tpu-jax"
if not (ENGINE_DIR / ".git").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/xbill9/tpu-jax", str(ENGINE_DIR)], check=True)
subprocess.run(["git", "-C", str(ENGINE_DIR), "checkout", "-q", "4b9f8e94619f1995206617e07e3c4ac62685d47b"], check=True)
sys.path.insert(0, str(ENGINE_DIR)); sys.path.insert(0, str(WORK))
(WORK / "gguf_jax.py").write_text('"""Load a Gemma 4 E2B Q4_0 GGUF straight onto the accelerator, as the tensors tpu-jax reads from safetensors.\n\nOnly the raw 4-bit blocks cross to the device (2.6 GB); each is expanded there to bf16 as\nbf16(q * d), the single rounding the engine\'s own W4A16 unpack gives.\n"""\nimport json\nimport re\nimport struct\nimport time\n\nimport numpy as np\nimport jax\nimport jax.numpy as jnp\n\nP = "model.language_model."\nTOP = {\n    "token_embd.weight": "embed_tokens.weight",\n    "per_layer_token_embd.weight": "embed_tokens_per_layer.weight",\n    "per_layer_model_proj.weight": "per_layer_model_projection.weight",\n    "per_layer_proj_norm.weight": "per_layer_projection_norm.weight",\n    "output_norm.weight": "norm.weight",\n}\nBLK = {\n    "attn_q": "self_attn.q_proj.weight", "attn_k": "self_attn.k_proj.weight",\n    "attn_v": "self_attn.v_proj.weight", "attn_output": "self_attn.o_proj.weight",\n    "attn_q_norm": "self_attn.q_norm.weight", "attn_k_norm": "self_attn.k_norm.weight",\n    "ffn_gate": "mlp.gate_proj.weight", "ffn_up": "mlp.up_proj.weight", "ffn_down": "mlp.down_proj.weight",\n    "inp_gate": "per_layer_input_gate.weight", "proj": "per_layer_projection.weight",\n    "attn_norm": "input_layernorm.weight", "post_attention_norm": "post_attention_layernorm.weight",\n    "ffn_norm": "pre_feedforward_layernorm.weight", "post_ffw_norm": "post_feedforward_layernorm.weight",\n    "post_norm": "post_per_layer_input_norm.weight", "layer_output_scale": "layer_scalar",\n}\nSKIP = {"rope_freqs.weight"}  # the engine computes RoPE from config.json\nF32, Q4_0 = 0, 2\nCHUNK_ROWS = 16384  # rows expanded per device call, so a big table never needs an f32 copy\n\n\ndef hf_name(g):\n    if g in TOP:\n        return P + TOP[g]\n    m = re.fullmatch(r"blk\\.(\\d+)\\.(.+)\\.weight", g)\n    if m and m.group(2) in BLK:\n        return f"{P}layers.{m.group(1)}.{BLK[m.group(2)]}"\n    return None\n\n\ndef read_index(path):\n    """GGUF v3 header -> (data offset, [(name, dims, type, offset)]). Metadata values are skipped."""\n    with open(path, "rb") as f:\n        r = lambda fmt: struct.unpack("<" + fmt, f.read(struct.calcsize("<" + fmt)))\n        if f.read(4) != b"GGUF":\n            raise ValueError(f"{path}: not a GGUF file")\n        r("I")\n        n_t, n_kv = r("QQ")\n        align = 32\n\n        def rs():\n            return f.read(r("Q")[0]).decode()\n\n        size = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}\n\n        def rv(t):\n            if t == 8:\n                return rs()\n            if t == 9:\n                at, n = r("IQ")\n                if at in size:\n                    f.seek(size[at] * n, 1)\n                    return None\n                return [rv(at) for _ in range(n)]\n            return r({0: "B", 1: "b", 2: "H", 3: "h", 4: "I", 5: "i", 6: "f", 7: "?", 10: "Q", 11: "q", 12: "d"}[t])[0]\n\n        for _ in range(n_kv):\n            k = rs()\n            v = rv(r("I")[0])\n            if k == "general.alignment":\n                align = v\n        tensors = []\n        for _ in range(n_t):\n            name = rs()\n            nd = r("I")[0]\n            dims = r("Q" * nd)\n            t, off = r("IQ")\n            tensors.append((name, dims, t, off))\n        p = f.tell()\n    return p + (-p % align), tensors\n\n\ndef _dequant(blocks):\n    """uint8 [rows, nb, 18] block_q4_0 -> bf16 [rows, nb*32]."""\n    lo, hi = blocks[..., 0].astype(jnp.uint16), blocks[..., 1].astype(jnp.uint16)\n    d = jax.lax.bitcast_convert_type(lo | (hi << 8), jnp.float16).astype(jnp.float32)[..., None]\n    qs = blocks[..., 2:]\n    q = jnp.concatenate([qs & 0xF, qs >> 4], axis=-1).astype(jnp.int32) - 8\n    v = (q.astype(jnp.float32) * d).astype(jnp.bfloat16)\n    return v.reshape(blocks.shape[0], -1)\n\n\n_dequant_jit = jax.jit(_dequant)\n\n\n@jax.jit\ndef _dequant_into(out, blocks, row):\n    return jax.lax.dynamic_update_slice(out, _dequant(blocks), (row, 0))\n\n\n_dequant_into = jax.jit(_dequant_into.__wrapped__, donate_argnums=0)\n\n\ndef load_gguf(path, device=None):\n    """GGUF -> {safetensors name: bf16 jax.Array on `device`}, HF [out, in] orientation."""\n    device = device or jax.devices()[0]\n    t0 = time.perf_counter()\n    data_start, tensors = read_index(path)\n    mm = np.memmap(path, np.uint8, "r")\n    out, nbytes = {}, 0\n    for name, dims, t, off in tensors:\n        if name in SKIP:\n            continue\n        h = hf_name(name)\n        if h is None:\n            raise KeyError(f"GGUF tensor {name} has no tpu-jax counterpart")\n        shape = tuple(int(x) for x in reversed(dims))\n        start = data_start + off\n        if t == F32:\n            n = int(np.prod(shape))\n            a = np.frombuffer(mm[start : start + 4 * n], np.float32).reshape(shape)\n            out[h] = jax.device_put(jnp.asarray(a, jnp.bfloat16), device)\n            nbytes += 4 * n\n        elif t == Q4_0:\n            rows, cols = shape\n            nb = cols // 32\n            blocks = mm[start : start + rows * nb * 18].reshape(rows, nb, 18)\n            nbytes += blocks.size\n            if rows <= CHUNK_ROWS:\n                out[h] = _dequant_jit(jax.device_put(np.ascontiguousarray(blocks), device))\n            else:\n                if rows % CHUNK_ROWS:\n                    raise ValueError(f"{name}: {rows} rows is not a multiple of {CHUNK_ROWS}")\n                w = jax.device_put(jnp.zeros((rows, cols), jnp.bfloat16), device)\n                for r0 in range(0, rows, CHUNK_ROWS):\n                    chunk = jax.device_put(np.ascontiguousarray(blocks[r0 : r0 + CHUNK_ROWS]), device)\n                    w = _dequant_into(w, chunk, r0)\n                out[h] = w\n        else:\n            raise TypeError(f"GGUF tensor {name}: type {t} is not F32 or Q4_0")\n    jax.block_until_ready(list(out.values()))\n    return out, {"tensors": len(out), "gguf_bytes_read": nbytes, "seconds": time.perf_counter() - t0}\n')
import huggingface_hub, platform
print("huggingface_hub", huggingface_hub.__version__, "| python", platform.python_version(), "| cpus", os.cpu_count())
R = {}
print(f"setup {time.perf_counter() - T0:.1f} s")


huggingface_hub 1.33.0 | python 3.13.18 | cpus 24
setup 1.3 s


In [ ]:
# B: GGUF route -- download the 2.62 GB GGUF and the -text repo's small files, in parallel
from huggingface_hub import hf_hub_download, snapshot_download
GGUF_REPO, GGUF_FILE, GGUF_REV = "xbill9/gemma-4-E2B-it-qat-q4_0-exact-gguf", "gemma-4-E2B-it-q4_0-exact.gguf", "74b2c92c8f7f84d3d9c39e8026f6a28f62788255"
ST_REPO, ST_REV = "xbill9/gemma-4-E2B-it-qat-q4_0-w4a16-ct-text", "315aec06cff196fd6479aa2669bf9379b0d0268a"
out = {}
def get_gguf():
    t = time.perf_counter(); out["gguf"] = hf_hub_download(GGUF_REPO, GGUF_FILE, revision=GGUF_REV, token=False); out["gguf_s"] = time.perf_counter() - t
def get_small():
    t = time.perf_counter(); out["small"] = snapshot_download(ST_REPO, revision=ST_REV, token=False, ignore_patterns=["*.safetensors"]); out["small_s"] = time.perf_counter() - t
t = time.perf_counter()
th = [threading.Thread(target=f) for f in (get_gguf, get_small)]; [x.start() for x in th]; [x.join() for x in th]
R["gguf download s"] = time.perf_counter() - t
small_bytes = sum(f.stat().st_size for f in pathlib.Path(out["small"]).iterdir())
print(f"GGUF {os.path.getsize(out['gguf'])/1e9:.2f} GB in {out['gguf_s']:.1f} s; small files {small_bytes/1e6:.1f} MB in {out['small_s']:.1f} s; both done in {R['gguf download s']:.1f} s")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

gemma-4-E2B-it-q4_0-exact.gguf: reconstructing file:   0%|          |  0.00B / 2.62GB            

gemma-4-E2B-it-q4_0-exact.gguf: downloading bytes:           |  0.00B            

GGUF 2.62 GB in 25.0 s; small files 32.3 MB in 1.3 s; both done in 25.0 s


In [ ]:
# C: load from the GGUF through the unchanged ENGINE.load, then greedy requests
t = time.perf_counter()
import jax, jax.numpy as jnp
dev = jax.devices()[0]
R["jax init s"] = time.perf_counter() - t
print(f"jax {jax.__version__}, {dev.device_kind}, init {R['jax init s']:.1f} s")
from jax_engine import JaxGemmaEngine
from transformers import AutoTokenizer
import gguf_jax, safetensors.flax as sfx

SMALL = pathlib.Path(out["small"])
TOK = AutoTokenizer.from_pretrained(SMALL)
EOS = json.loads((SMALL / "generation_config.json").read_text())["eos_token_id"]

def make_gguf_dir():
    d = WORK / "gguf-model"; d.mkdir(exist_ok=True)
    for f in SMALL.iterdir():
        (d / f.name).unlink(missing_ok=True); (d / f.name).symlink_to(f.resolve())
    (d / "from-gguf.safetensors").write_bytes(b"")  # placeholder: ENGINE.load lists *.safetensors
    return d

def load_from_gguf(engine, gguf_path):
    d = make_gguf_dir()
    raw, info = gguf_jax.load_gguf(gguf_path)
    orig = sfx.load_file
    sfx.load_file = lambda p, *a, **k: raw if p.endswith("from-gguf.safetensors") else orig(p, *a, **k)
    try:
        engine.load(local_dir=str(d))
    finally:
        sfx.load_file = orig
    return info

PROMPTS = [
    ("Why is the sky blue? One paragraph.", False, 512),
    ("A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?", False, 512),
    ("Explain a key-value cache in a transformer, simply.", True, 2048),
]
def run_prompts(engine):
    res = []
    for text, think, budget in PROMPTS:
        enc = TOK.apply_chat_template([{"role": "user", "content": text}], tokenize=True, add_generation_prompt=True, enable_thinking=think)
        ids = list(enc["input_ids"] if hasattr(enc, "keys") else enc)
        t = time.perf_counter(); toks = []
        for item in engine.generate_stream(ids, budget, 0.0, -1, EOS, 0):
            if not isinstance(item, int): break
            toks.append(item)
        res.append({"prompt": text, "thinking": think, "tokens": toks, "seconds": time.perf_counter() - t})
        print(f"  {len(toks):4d} tokens in {res[-1]['seconds']:5.1f} s  {TOK.decode(toks, skip_special_tokens=True)[-90:]!r}")
    return res

t = time.perf_counter()
E = JaxGemmaEngine(ST_REPO, quant_mode="w4a16", max_model_len=4096, dequant_at_load=True)
info = load_from_gguf(E, out["gguf"])
E.bos_token_id = TOK.bos_token_id
R["gguf load s"] = time.perf_counter() - t
R["gguf to device s"] = info["seconds"]
print(f"GGUF -> chip {info['seconds']:.1f} s ({info['tensors']} tensors, {info['gguf_bytes_read']/1e9:.2f} GB read); whole load {R['gguf load s']:.1f} s; "
      f"{E.weight_bytes/1e9:.2f} GB of weights; HBM in use {dev.memory_stats()['bytes_in_use']/1e9:.2f} GB")
R["gguf ready s (from T0, excl. setup)"] = R["gguf download s"] + R["jax init s"] + R["gguf load s"]
print("greedy, GGUF:")
RES_G = run_prompts(E)


/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


jax 0.11.2, TPU v5 lite, init 7.1 s


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


GGUF -> chip 8.2 s (540 tensors, 2.60 GB read); whole load 9.2 s; 9.26 GB of weights; HBM in use 9.27 GB
greedy, GGUF:


   117 tokens in  25.0 s  ' Because blue light is scattered across the entire sky, our eyes perceive the sky as blue.'


   391 tokens in   3.1 s  '0 = \\$1.05$.\nTotal cost: $\\$1.05 + \\$0.05 = \\$1.10$. (This matches the problem statement.)'


   977 tokens in  29.9 s  'nderstanding long text much **faster** and **more efficient** without losing any accuracy.'


In [ ]:
# D: free the chip, then the current route: the 6.59 GB safetensors checkpoint
del E; gc.collect()
print(f"HBM in use after freeing: {dev.memory_stats()['bytes_in_use']/1e9:.2f} GB")
t = time.perf_counter()
ST_DIR = snapshot_download(ST_REPO, revision=ST_REV, token=False)
R["safetensors download s"] = time.perf_counter() - t
sz = sum(p.stat().st_size for p in pathlib.Path(ST_DIR).glob("*.safetensors"))
print(f"safetensors {sz/1e9:.2f} GB in {R['safetensors download s']:.1f} s")
t = time.perf_counter()
E = JaxGemmaEngine(ST_REPO, quant_mode="w4a16", max_model_len=4096, dequant_at_load=True)
E.load(local_dir=ST_DIR); E.bos_token_id = TOK.bos_token_id
R["safetensors load s"] = time.perf_counter() - t
print(f"safetensors load {R['safetensors load s']:.1f} s; {E.weight_bytes/1e9:.2f} GB of weights; HBM in use {dev.memory_stats()['bytes_in_use']/1e9:.2f} GB")
print("greedy, safetensors:")
RES_S = run_prompts(E)


HBM in use after freeing: 0.04 GB


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 47 files:   0%|          | 0/47 [00:00<?, ?it/s]

safetensors 6.56 GB in 62.7 s


safetensors load 16.9 s; 9.26 GB of weights; HBM in use 9.31 GB
greedy, safetensors:


   123 tokens in   2.2 s  'across the entire sky, it reaches our eyes from every direction, making the sky look blue.'


   391 tokens in   3.0 s  '0 = \\$1.05$.\nTotal cost: $\\$1.05 + \\$0.05 = \\$1.10$. (This matches the problem statement.)'


   995 tokens in   9.2 s  'nd more efficient** without having to waste time recalculating the same things repeatedly.'


In [ ]:
# E: compare
for g, s in zip(RES_G, RES_S):
    a, b = g["tokens"], s["tokens"]
    k = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
    print(f"{g['prompt'][:45]!r:50s} thinking={g['thinking']!s:5s} GGUF {len(a):4d} tok, safetensors {len(b):4d} tok, "
          f"identical={a == b}, first {k} tokens agree")
for k, v in R.items():
    print(f"  {k:40s} {v:6.1f}")
json.dump({"timings": R, "gguf": RES_G, "safetensors": RES_S}, open(WORK / "result.json", "w"))


'Why is the sky blue? One paragraph.'              thinking=False GGUF  117 tok, safetensors  123 tok, identical=False, first 109 tokens agree
'A bat and a ball cost $1.10 in total. The bat'    thinking=False GGUF  391 tok, safetensors  391 tok, identical=True, first 391 tokens agree
'Explain a key-value cache in a transformer, s'    thinking=True  GGUF  977 tok, safetensors  995 tok, identical=False, first 718 tokens agree
  gguf download s                            25.0
  jax init s                                  7.1
  gguf load s                                 9.2
  gguf to device s                            8.2
  gguf ready s (from T0, excl. setup)        41.3
  safetensors download s                     62.7
  safetensors load s                         16.9
